# 16 — Custom gradients and systematic debugging

**Level:** Advanced / expert practice · **Study time:** 75–100 minutes (including experiments)

**Prerequisites:** Notebooks 03, 05, and 08.

## What you will be able to do
- Implement a custom autograd Function.
- Validate first and second derivatives.
- Investigate a broken gradient and overfit a tiny batch.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 15](15_checkpoints_and_inference.ipynb) · [Next: 17](17_performance.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Implement a derivative explicitly

Most models should compose existing PyTorch operations; autograd already differentiates them. A custom Function is useful for unusual operations or specialized implementations. Backward applies the chain rule: multiply the local derivative by the incoming gradient.


In [ ]:
class Cube(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.pow(3)
    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 3 * x.square()

x = torch.tensor([1., 2., -3.], requires_grad=True)
Cube.apply(x).sum().backward()
assert torch.allclose(x.grad, 3 * x.detach().square())
print("Custom gradients:", x.grad)


## Step 3 — Check first and second derivatives

A correct forward result does not establish a correct backward implementation. Gradcheck compares analytical gradients with finite differences. Use double precision and smooth points; finite differences are unreliable at nondifferentiable boundaries.


In [ ]:
probe = torch.randn(5, dtype=torch.float64, requires_grad=True)
assert torch.autograd.gradcheck(Cube.apply, (probe,))
assert torch.autograd.gradgradcheck(Cube.apply, (probe,))
print("First- and second-order gradient checks passed.")

class WrongCube(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.pow(3)
    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x  # Deliberate bug: derivative of x^2, not x^3.

wrong_passes = torch.autograd.gradcheck(WrongCube.apply, (probe,), raise_exception=False)
assert not wrong_passes
print("Deliberately wrong backward passed?", wrong_passes)


## Step 4 — Inspect gradient flow

A missing gradient can indicate an unused parameter, a detached tensor, or a disabled graph. Check gradients immediately after backward. Do not infer gradient health from the loss alone.


In [ ]:
model = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 1))
xb, yb = torch.randn(8, 3), torch.randn(8, 1)
loss = nn.functional.mse_loss(model(xb), yb)
loss.backward()
for name, parameter in model.named_parameters():
    assert parameter.grad is not None
    assert torch.isfinite(parameter.grad).all()
    print(name, "gradient norm:", parameter.grad.norm().item())
detached_prediction = model(xb).detach()
assert not detached_prediction.requires_grad


## Step 5 — Overfit a tiny batch as a diagnostic

Before a long training run, verify a sufficiently flexible model can memorize a handful of examples with regularization disabled. Failure suggests checking labels, loss/activation compatibility, learning rate, parameter registration, and gradient flow. Success is a plumbing check, not proof of generalization.


In [ ]:
torch.manual_seed(SEED)
tiny_x, tiny_y = torch.randn(8, 3), torch.randn(8, 1)
tiny = nn.Sequential(nn.Linear(3, 32), nn.Tanh(), nn.Linear(32, 1))
optimizer = torch.optim.Adam(tiny.parameters(), lr=0.03)
for step in range(400):
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.mse_loss(tiny(tiny_x), tiny_y)
    assert torch.isfinite(loss)
    loss.backward(); optimizer.step()
with torch.no_grad():
    final_loss = nn.functional.mse_loss(tiny(tiny_x), tiny_y).item()
print("Tiny-batch MSE:", final_loss)
assert final_loss < 1e-3


## Your turn — practice before opening the solutions

1. Implement a custom square function and run gradcheck.
2. Compute a Hessian for `sum(x³)` with `torch.autograd.functional.hessian`.
3. Clip the tiny model's gradients and verify their total norm is at most 0.1. Hint: clip after backward and before the optimizer step.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
class Square(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x.square()
    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return grad_output * 2 * x
assert torch.autograd.gradcheck(Square.apply, (probe,))
hessian = torch.autograd.functional.hessian(lambda z: z.pow(3).sum(), probe)
assert torch.allclose(hessian, torch.diag(6 * probe))
optimizer.zero_grad(set_to_none=True)
nn.functional.mse_loss(tiny(tiny_x), tiny_y + 10).backward()
nn.utils.clip_grad_norm_(tiny.parameters(), 0.1)
total_norm = torch.stack([p.grad.norm() for p in tiny.parameters()]).norm()
assert total_norm <= 0.10001


## Before moving on

Debug in this order: shapes and dtypes, input/target alignment, finite loss, gradient presence, tiny-batch fit, then generalization. Use anomaly detection around forward and backward when tracking a difficult NaN; it is slower and is a diagnostic tool.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Extending autograd](https://docs.pytorch.org/docs/stable/notes/extending.html)
- [Gradcheck mechanics](https://docs.pytorch.org/docs/stable/notes/gradcheck.html)


[Course guide](../README.md) · [Previous: 15](15_checkpoints_and_inference.ipynb) · [Next: 17](17_performance.ipynb)
